# Synthetic Data Fidelity Evaluation

Measures how closely LLM-augmented data resembles the original UIT-VSMEC training set.

**Fidelity components:**
1. **Statistical similarity** — summary statistics and distribution tests
2. **Correlation preservation** — relationships between numeric/text features

For lexical **diversity** (Distinct-n, Self-BLEU), see `Generated_data_eval.ipynb`.


In [ ]:
%pip install -q pandas numpy scipy scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import warnings

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_DIR = Path("data/processed")
EVAL_FILE = DATA_DIR / "train_1500_gen_eval.csv"
ORIG_FILE = DATA_DIR / "train_org_processed.csv"
TEXT_COL = "Sentence"
EMOTION_COL = "Emotion"


## Helper functions


In [ ]:
def word_count(text: str) -> int:
    if pd.isna(text) or not str(text).strip():
        return 0
    return len(str(text).split())


def compute_sentence_stats(texts) -> dict:
    lengths = [len(str(t)) for t in texts if pd.notna(t)]
    if not lengths:
        return {"mean": 0, "median": 0, "min": 0, "max": 0, "std": 0, "var": 0}
    arr = np.array(lengths, dtype=float)
    return {
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "min": float(arr.min()),
        "max": float(arr.max()),
        "std": float(arr.std()),
        "var": float(arr.var()),
    }


def word_count_stats(texts) -> dict:
    counts = [word_count(t) for t in texts if pd.notna(t)]
    if not counts:
        return {"mean": 0, "median": 0, "min": 0, "max": 0, "std": 0, "var": 0}
    arr = np.array(counts, dtype=float)
    return {
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "min": float(arr.min()),
        "max": float(arr.max()),
        "std": float(arr.std()),
        "var": float(arr.var()),
    }


def analyze_emotion_distribution(df: pd.DataFrame, emotion_col: str = "Emotion") -> dict:
    emotion_counts = df[emotion_col].value_counts()
    total = len(df)
    entropy = 0.0
    for count in emotion_counts:
        if count > 0:
            p = count / total
            entropy -= p * np.log2(p)
    return {
        "distribution": emotion_counts.to_dict(),
        "total_samples": total,
        "num_classes": len(emotion_counts),
        "entropy": entropy,
    }


def compute_statistical_comparison(dist_orig: dict, dist_synth: dict) -> dict:
    c1, c2 = dist_orig["distribution"], dist_synth["distribution"]
    all_emotions = set(c1) | set(c2)
    t1, t2 = dist_orig["total_samples"], dist_synth["total_samples"]
    total_both = t1 + t2
    chi_square = 0.0
    for emotion in all_emotions:
        n1, n2 = c1.get(emotion, 0), c2.get(emotion, 0)
        te = n1 + n2
        e1 = te * t1 / total_both if total_both else 0
        e2 = te * t2 / total_both if total_both else 0
        if e1 > 0:
            chi_square += (n1 - e1) ** 2 / e1
        if e2 > 0:
            chi_square += (n2 - e2) ** 2 / e2
    kl_div = 0.0
    for emotion in all_emotions:
        p = c1.get(emotion, 0) / t1 if t1 else 0
        q = c2.get(emotion, 0) / t2 if t2 else 0
        if p > 0 and q > 0:
            kl_div += p * np.log2(p / q)
    kl_div = max(kl_div, 0.0)
    return {"chi_square_statistic": chi_square, "kl_divergence": kl_div}


def relative_error(orig_val: float, synth_val: float) -> float:
    if orig_val == 0:
        return abs(synth_val)
    return abs(synth_val - orig_val) / abs(orig_val)


def stat_similarity_score(orig_stats: dict, synth_stats: dict) -> float:
    keys = ["mean", "std"]
    scores = []
    for k in keys:
        scores.append(1.0 - min(relative_error(orig_stats[k], synth_stats[k]), 1.0))
    return float(np.mean(scores)) if scores else 0.0


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    texts = df[TEXT_COL].astype(str)
    wc = texts.map(word_count)
    char_len = texts.map(len)
    avg_wlen = wc.replace(0, np.nan)
    avg_wlen = (char_len / avg_wlen).fillna(0)
    out = pd.DataFrame({"char_len": char_len, "word_count": wc, "avg_word_len": avg_wlen})
    dummies = pd.get_dummies(df[EMOTION_COL].astype(str), prefix="emo")
    return pd.concat([out, dummies], axis=1)


def corr_matrix_triangle_pearson(c1: pd.DataFrame, c2: pd.DataFrame) -> float:
    common = c1.columns.intersection(c2.columns)
    c1, c2 = c1.loc[common, common], c2.loc[common, common]
    iu = np.triu_indices(len(common), k=1)
    a, b = c1.values[iu], c2.values[iu]
    mask = ~(np.isnan(a) | np.isnan(b))
    if mask.sum() < 2:
        return 0.0
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def normalized_frobenius(c1: pd.DataFrame, c2: pd.DataFrame) -> float:
    common = c1.columns.intersection(c2.columns)
    c1 = c1.loc[common, common].fillna(0)
    c2 = c2.loc[common, common].fillna(0)
    diff = (c1 - c2).values
    return float(np.linalg.norm(diff, "fro") / max(diff.size ** 0.5, 1e-9))


## 1. Load and split data


In [ ]:
def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run Data_Augmentation_LLM.ipynb export cells first."
        )

require_file(EVAL_FILE)
require_file(ORIG_FILE)

df_eval = pd.read_csv(EVAL_FILE)
df_orig = pd.read_csv(ORIG_FILE)

if "augmented" not in df_eval.columns:
    raise KeyError("eval CSV must include 'augmented' column — use train_1500_gen_eval.csv")

df_synth = df_eval[df_eval["augmented"] == True].copy()
df_orig_ref = df_orig.copy()

print(f"Eval file: {len(df_eval):,} rows, columns: {list(df_eval.columns)}")
print(f"Synthetic (augmented=True): {len(df_synth):,}")
print(f"Original reference: {len(df_orig_ref):,}")
print(f"Split method: augmented flag in {EVAL_FILE.name}")

assert len(df_synth) > 0, "No synthetic rows found"
assert len(df_orig_ref) > 0, "Original reference is empty"

display(df_eval["augmented"].value_counts().to_frame("count"))


## 2. Statistical similarity


In [ ]:
orig_texts = df_orig_ref[TEXT_COL].tolist()
synth_texts = df_synth[TEXT_COL].tolist()

char_orig = compute_sentence_stats(orig_texts)
char_synth = compute_sentence_stats(synth_texts)
wc_orig = word_count_stats(orig_texts)
wc_synth = word_count_stats(synth_texts)

ks_char = stats.ks_2samp(
    [len(str(t)) for t in orig_texts if pd.notna(t)],
    [len(str(t)) for t in synth_texts if pd.notna(t)],
)
ks_wc = stats.ks_2samp(
    [word_count(t) for t in orig_texts if pd.notna(t)],
    [word_count(t) for t in synth_texts if pd.notna(t)],
)

summary_rows = []
for name, o, s in [
    ("char_length_mean", char_orig["mean"], char_synth["mean"]),
    ("char_length_median", char_orig["median"], char_synth["median"]),
    ("char_length_std", char_orig["std"], char_synth["std"]),
    ("char_length_var", char_orig["var"], char_synth["var"]),
    ("word_count_mean", wc_orig["mean"], wc_synth["mean"]),
    ("word_count_median", wc_orig["median"], wc_synth["median"]),
    ("word_count_std", wc_orig["std"], wc_synth["std"]),
    ("word_count_var", wc_orig["var"], wc_synth["var"]),
]:
    summary_rows.append({
        "metric": name,
        "original": o,
        "synthetic": s,
        "abs_diff": abs(s - o),
        "rel_error": relative_error(o, s),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print(f"KS test (char length): statistic={ks_char.statistic:.4f}, p-value={ks_char.pvalue:.4f}")
print(f"KS test (word count): statistic={ks_wc.statistic:.4f}, p-value={ks_wc.pvalue:.4f}")

emo_orig = analyze_emotion_distribution(df_orig_ref)
emo_synth = analyze_emotion_distribution(df_synth)
emo_cmp = compute_statistical_comparison(emo_orig, emo_synth)
print(f"Emotion chi-square (orig vs synth): {emo_cmp['chi_square_statistic']:.4f}")
print(f"Emotion KL divergence (orig || synth): {emo_cmp['kl_divergence']:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

orig_char = [len(str(t)) for t in orig_texts if pd.notna(t)]
synth_char = [len(str(t)) for t in synth_texts if pd.notna(t)]
axes[0].hist(orig_char, bins=40, alpha=0.5, density=True, label="Original")
axes[0].hist(synth_char, bins=40, alpha=0.5, density=True, label="Synthetic")
axes[0].set_title("Character length distribution")
axes[0].legend()

orig_wc = [word_count(t) for t in orig_texts if pd.notna(t)]
synth_wc = [word_count(t) for t in synth_texts if pd.notna(t)]
axes[1].hist(orig_wc, bins=40, alpha=0.5, density=True, label="Original")
axes[1].hist(synth_wc, bins=40, alpha=0.5, density=True, label="Synthetic")
axes[1].set_title("Word count distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

# Emotion proportions
emo_labels = sorted(set(emo_orig["distribution"]) | set(emo_synth["distribution"]))
orig_props = [emo_orig["distribution"].get(e, 0) / emo_orig["total_samples"] for e in emo_labels]
synth_props = [emo_synth["distribution"].get(e, 0) / emo_synth["total_samples"] for e in emo_labels]
x = np.arange(len(emo_labels))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width / 2, orig_props, width, label="Original")
ax.bar(x + width / 2, synth_props, width, label="Synthetic")
ax.set_xticks(x)
ax.set_xticklabels(emo_labels, rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_title("Emotion distribution")
ax.legend()
plt.tight_layout()
plt.show()


## 3. Correlation preservation


In [ ]:
feat_orig = build_feature_frame(df_orig_ref)
feat_synth = build_feature_frame(df_synth)

# Align columns (synthetic subset may miss rare emotion dummies)
all_cols = feat_orig.columns.union(feat_synth.columns)
feat_orig = feat_orig.reindex(columns=all_cols, fill_value=0)
feat_synth = feat_synth.reindex(columns=all_cols, fill_value=0)

C_orig = feat_orig.corr(method="pearson")
C_synth = feat_synth.corr(method="pearson")

corr_r = corr_matrix_triangle_pearson(C_orig, C_synth)
fro_norm = normalized_frobenius(C_orig, C_synth)

print(f"Correlation matrix triangle Pearson r: {corr_r:.4f}")
print(f"Normalized Frobenius norm of difference: {fro_norm:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(C_orig, ax=axes[0], cmap="coolwarm", center=0, vmin=-1, vmax=1)
axes[0].set_title("Original correlations")
sns.heatmap(C_synth, ax=axes[1], cmap="coolwarm", center=0, vmin=-1, vmax=1)
axes[1].set_title("Synthetic correlations")
sns.heatmap(C_synth - C_orig, ax=axes[2], cmap="coolwarm", center=0)
axes[2].set_title("Difference (synth - orig)")
plt.tight_layout()
plt.show()


## 4. Composite fidelity score

Equal-weight average of four components (each in [0, 1], higher is better):

| Component | Description |
|-----------|-------------|
| `stat_similarity` | 1 − relative error on mean/std of length & word count |
| `distribution_similarity` | Mean of 1/(1 + KS statistic) for char length and word count |
| `label_similarity` | exp(−KL divergence) on emotion distribution |
| `correlation_similarity` | Mean of triangle Pearson r and (1 − normalized Frobenius) |

**Guidance (non-blocking):** correlation r > 0.85, KS p > 0.05, fidelity_score > 0.7 suggest good alignment.


In [ ]:
stat_sim = np.mean([
    stat_similarity_score(char_orig, char_synth),
    stat_similarity_score(wc_orig, wc_synth),
])

dist_sim = np.mean([
    1.0 / (1.0 + ks_char.statistic),
    1.0 / (1.0 + ks_wc.statistic),
])

label_sim = float(np.exp(-emo_cmp["kl_divergence"]))
fro_sim = max(0.0, 1.0 - min(fro_norm, 1.0))
corr_sim = 0.5 * max(corr_r, 0.0) + 0.5 * fro_sim

components = {
    "stat_similarity": stat_sim,
    "distribution_similarity": dist_sim,
    "label_similarity": label_sim,
    "correlation_similarity": corr_sim,
}
fidelity_score = float(np.mean(list(components.values())))

fidelity_df = pd.DataFrame([
    {"component": k, "score": v} for k, v in components.items()
])
fidelity_df.loc[len(fidelity_df)] = {"component": "fidelity_score (mean)", "score": fidelity_score}

print("=== Fidelity components ===")
display(fidelity_df)

print(f"\nOverall fidelity score: {fidelity_score:.4f}")


## 5. Summary table


In [ ]:
final_summary = summary_df.copy()
final_summary["sub_score_stat"] = 1 - final_summary["rel_error"].clip(0, 1)

print("=== Statistical summary ===")
display(final_summary)

print("=== Emotion & correlation ===")
display(pd.DataFrame([
    {"metric": "chi_square", "value": emo_cmp["chi_square_statistic"]},
    {"metric": "kl_divergence", "value": emo_cmp["kl_divergence"]},
    {"metric": "corr_triangle_pearson", "value": corr_r},
    {"metric": "frobenius_norm", "value": fro_norm},
    {"metric": "fidelity_score", "value": fidelity_score},
]))
